# 🧠 Notebook 3: LSTM Residual Correction & Hybrid Model
**Project:** A Hybrid Deep Learning Approach for Modelling Global CO₂ Emissions  
**Author:** Hafiza Alishba Naaz | NUST Islamabad 2026

> ⚠️ **Run Notebooks 1 & 2 first** so that `all_models`, `all_forecasts`, `top10_countries` are in memory.

## 3.1 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

print('✅ Imports loaded.')
print(f'TensorFlow version: {tf.__version__}')

## 3.2 LSTM Configuration

In [ ]:
# Best hyperparameters (from tuning)
LOOKBACK     = 3    # 3-year sliding window
LSTM_UNITS   = 32   # first LSTM layer units
DROPOUT_RATE = 0.2  # dropout between LSTM layers
LSTM_EPOCHS  = 200
LSTM_BATCH   = 4
PATIENCE     = 20

print('LSTM Configuration:')
print(f'  Lookback window : {LOOKBACK} years')
print(f'  LSTM units      : {LSTM_UNITS}')
print(f'  Dropout rate    : {DROPOUT_RATE}')
print(f'  Max epochs      : {LSTM_EPOCHS}')
print(f'  Batch size      : {LSTM_BATCH}')

## 3.3 Helper Functions

In [ ]:
def create_sequences(data, lookback):
    """Convert residual series to supervised learning sequences."""
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i])
        y.append(data[i])
    return np.array(X).reshape(-1, lookback, 1), np.array(y)

def build_lstm_model(lookback, units, dropout_rate):
    """Build LSTM architecture: LSTM(units) -> LSTM(units//2) -> Dense(1)"""
    model = Sequential([
        Input(shape=(lookback, 1)),
        LSTM(units, return_sequences=True),
        Dropout(dropout_rate),
        LSTM(max(units // 2, 8)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

print('✅ Helper functions defined.')
build_lstm_model(LOOKBACK, LSTM_UNITS, DROPOUT_RATE).summary()

## 3.4 Train Hybrid Model — All 10 Countries

In [ ]:
# NOTE: This cell requires all_models and all_forecasts from Notebook 2.
# If running standalone, load them from saved files.

y_col = 'Value_co2_emissions_kt_by_country'
hybrid_results = []
hybrid_forecasts = {}

print('Training Hybrid SARIMAX + LSTM for all 10 countries...')
print('='*70)

for country in top10_countries:
    sarimax_fit        = all_models[country]
    fc_dict            = all_forecasts[country]
    test_actual        = fc_dict['test'][y_col]
    sarimax_preds_test = fc_dict['preds']
    train_resid        = sarimax_fit.resid.dropna()

    if len(train_resid) <= LOOKBACK:
        print(f'{country}: Insufficient residuals, skipping.')
        continue

    # Scale residuals to [-1, 1]
    resid_scaler = MinMaxScaler(feature_range=(-1, 1))
    resid_scaled = resid_scaler.fit_transform(train_resid.values.reshape(-1,1)).flatten()

    X_seq, y_seq = create_sequences(resid_scaled, LOOKBACK)
    split = max(1, int(len(X_seq) * 0.8))
    X_tr_r, X_val_r = X_seq[:split], X_seq[split:]
    y_tr_r, y_val_r = y_seq[:split], y_seq[split:]
    val_data = (X_val_r, y_val_r) if len(X_val_r) > 0 else None

    model = build_lstm_model(LOOKBACK, LSTM_UNITS, DROPOUT_RATE)
    es    = EarlyStopping(monitor='val_loss' if val_data else 'loss',
                          patience=PATIENCE, restore_best_weights=True, verbose=0)
    model.fit(X_tr_r, y_tr_r, validation_data=val_data,
              epochs=LSTM_EPOCHS, batch_size=LSTM_BATCH,
              callbacks=[es], verbose=0)

    # Autoregressive residual forecast for test period
    seed_seq = resid_scaled[-LOOKBACK:].tolist()
    pred_resid_scaled = []
    for _ in range(len(test_actual)):
        inp  = np.array(seed_seq[-LOOKBACK:]).reshape(1, LOOKBACK, 1)
        pred = model.predict(inp, verbose=0)[0, 0]
        pred_resid_scaled.append(pred)
        seed_seq.append(pred)

    pred_resid = resid_scaler.inverse_transform(
        np.array(pred_resid_scaled).reshape(-1,1)).flatten()

    hybrid_preds = pd.Series(sarimax_preds_test.values + pred_resid, index=test_actual.index)

    actual        = test_actual.values
    hybrid        = hybrid_preds.values
    sarimax_only  = sarimax_preds_test.values

    mape_hybrid   = np.mean(np.abs((actual - hybrid) / actual)) * 100
    mape_sarimax  = np.mean(np.abs((actual - sarimax_only) / actual)) * 100
    rmse_hybrid   = np.sqrt(mean_squared_error(actual, hybrid))
    r2_hybrid     = r2_score(actual, hybrid)

    hybrid_results.append({
        'Country': country,
        'SARIMAX_MAPE_%': round(mape_sarimax, 2),
        'Hybrid_MAPE_%':  round(mape_hybrid, 2),
        'MAPE_Improvement': round(mape_sarimax - mape_hybrid, 2),
        'Hybrid_RMSE': round(rmse_hybrid, 0),
        'Hybrid_R2':   round(r2_hybrid, 4),
    })
    hybrid_forecasts[country] = {'test': test_actual, 'sarimax': sarimax_preds_test, 'hybrid': hybrid_preds}

    print(f'{country:20s} | SARIMAX MAPE={mape_sarimax:.2f}% | Hybrid MAPE={mape_hybrid:.2f}% | R²={r2_hybrid:.4f}')
    tf.keras.backend.clear_session()

hybrid_df = pd.DataFrame(hybrid_results)
print('\n' + '='*70)
print(f'Mean SARIMAX MAPE : {hybrid_df["SARIMAX_MAPE_%"].mean():.2f}%')
print(f'Mean Hybrid MAPE  : {hybrid_df["Hybrid_MAPE_%"].mean():.2f}%')
print(f'Mean Hybrid R²    : {hybrid_df["Hybrid_R2"].mean():.4f}')
improved = (hybrid_df['MAPE_Improvement'] > 0).sum()
print(f'Countries improved: {improved}/{len(hybrid_df)}')

hybrid_df.to_csv('../results/hybrid_results.csv', index=False)
print('\n✅ Saved: results/hybrid_results.csv')

## 3.5 Hybrid vs SARIMAX — Forecast Comparison Plots

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
colors_plot = ['#378ADD','#1D9E75','#D85A30','#7F77DD','#BA7517',
               '#533AB7','#0F6E56','#993C1D','#639922','#3C3489']

for ax, country, color in zip(axes.flatten(), top10_countries, colors_plot):
    if country not in hybrid_forecasts: continue
    fc = hybrid_forecasts[country]
    r  = hybrid_df[hybrid_df['Country'] == country].iloc[0]

    ax.plot(fc['test'].index, fc['test'].values,   color=color,     lw=2, marker='o', markersize=5, label='Actual')
    ax.plot(fc['sarimax'].index, fc['sarimax'].values, color='#AAAAAA', lw=2, marker='s', markersize=4, linestyle='--', label='SARIMAX')
    ax.plot(fc['hybrid'].index,  fc['hybrid'].values,  color='#D85A30', lw=2, marker='^', markersize=5, linestyle='-',  label='Hybrid')
    ax.set_title(f"{country}\nHybrid MAPE={r['Hybrid_MAPE_%']:.1f}%", fontsize=8)
    ax.set_xlabel('Year', fontsize=7); ax.tick_params(labelsize=7)
    ax.grid(alpha=0.3); ax.legend(fontsize=6)

plt.suptitle('Hybrid SARIMAX+LSTM vs Standalone SARIMAX — Top 10 Emitters', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/hybrid_vs_sarimax.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 Ljung-Box Residual Diagnostics — Hybrid Model

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

print('Ljung-Box White Noise Test (p > 0.05 = white noise ✅)')
print('='*50)
for country in top10_countries:
    if country not in hybrid_forecasts: continue
    fc     = hybrid_forecasts[country]
    resid  = fc['test'].values - fc['hybrid'].values
    lb     = acorr_ljungbox(resid, lags=[5], return_df=True)
    lb_p   = lb['lb_pvalue'].values[0]
    result = '✅ White noise' if lb_p > 0.05 else '❌ Autocorrelation present'
    print(f'{country:20s} | p={lb_p:.3f} | {result}')